# H22b: Extreme Anisotropy -- Polar Factor Wastes Budget on Noise SVs

## Theoretical Motivation

The Newton-Schulz orthogonalization at the heart of Muon computes the polar factor
$UV^T$ of the gradient $G = U\Sigma V^T$, setting **all** singular values to 1.
This is beneficial when all singular directions carry meaningful loss information at
different magnitudes -- equalization makes the optimizer treat them equally.

But what happens when the gradient is **effectively low-rank**? If $G$ has effective
rank $k \ll d$, then the bottom $d-k$ singular values are essentially noise. The polar
factor still sets them to 1, which means:

$$\frac{\|UV^T\|_{\text{noise}}^2}{\|UV^T\|_F^2} = \frac{d - k}{d}$$

For a near-rank-1 gradient ($k = 1$, $d = 32$), this means **97% of the update's
Frobenius norm goes to noise directions**. The optimizer is spending almost all its
learning rate budget moving in random directions that carry no loss information.

## Hypothesis

> **H22b:** At extreme gradient anisotropy ($\kappa = \sigma_1/\sigma_{\min} \geq 1000$):
>
> 1. **T1:** The noise fraction of $\text{ortho}(G)$ is high ($> 50\%$), and higher
>    noise fraction correlates negatively with Muon's advantage ($r < -0.3$).
> 2. **T2:** A "Muon-clip" variant that zeros out the bottom $d-k$ singular values
>    of $G$ before orthogonalization recovers (or exceeds) Muon's advantage at high $\kappa$.

## The Muon-Clip Fix

Standard Muon: $\text{update} = \text{NS}(G) = UV^T$ (all SVs = 1)

Muon-clip(k): $\text{update} = \text{NS}(G_{\text{clip}})$ where $G_{\text{clip}}$ keeps
only the top-$k$ singular values of $G$ (zeroing the rest), then orthogonalizes. This
ensures the polar factor only equalizes **signal** directions, not noise.

## Experimental Design

| $\kappa$ (data condition) | Expected Eff. Rank | Expected Noise Frac | Expected Muon vs SGD |
|---------------------------|--------------------|--------------------|----------------------|
| 1                         | ~32 (full)         | ~0%                | Advantage ~1x        |
| 10                        | ~20-25             | ~25-35%            | Moderate advantage   |
| 100                       | ~10-15             | ~50-60%            | Declining advantage  |
| 1000                      | ~5-8               | ~75-85%            | Possible loss        |
| 10000                     | ~2-4               | ~90%+              | Muon may hurt        |

**Network:** Single-layer 32x32 linear, 500 steps, independent LR sweeps.
**Three optimizers compared:** SGD, Muon, Muon-clip(k).

## Environment Setup

Pure NumPy implementation. Single-layer linear network makes the gradient-to-data
relationship analytically transparent: $G = (W x - y)x^T / N$, so the gradient spectrum
directly inherits the data's condition number.

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

- **DIM = 32**: Single 32x32 weight matrix (32 singular values to analyze).
- **NUM_STEPS = 500**: Training iterations -- enough for convergence at this scale.
- **MOMENTUM = 0.9**: Identical for all three optimizers (SGD, Muon, Muon-clip).
- **NS_ITERS = 5**: Newton-Schulz iterations for polar decomposition.
- **NUM_SEEDS = 5**: Independent random seeds.
- **BATCH_SIZE = 64**: Fixed batch.

The single-layer design is intentional: it isolates the gradient-level noise amplification
effect without depth-related confounds (vanishing gradients, layer interactions).

In [ ]:
DIM = 32
NUM_STEPS = 500
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  DIM            = {DIM}    (single-layer weight matrix)")
print(f"  NUM_STEPS      = {NUM_STEPS}   (training iterations per run)")
print(f"  BATCH_SIZE     = {BATCH_SIZE}    (fixed batch)")
print(f"  MOMENTUM       = {MOMENTUM}  (same for SGD, Muon, Muon-clip)")
print(f"  NUM_SEEDS      = {NUM_SEEDS}     (independent random seeds)")
print(f"  NS_ITERS       = {NS_ITERS}     (Newton-Schulz iterations)")

### Anisotropy Control via Data Condition Number

The key experimental variable is $\kappa$, the condition number of the input data matrix.
Since the gradient of a single-layer linear network is $G = (Wx - y)x^T/N$, the gradient's
spectral structure is directly shaped by $x$'s condition number. Higher $\kappa$ means
the gradient is concentrated in fewer singular directions, creating the extreme anisotropy
we want to stress-test.

In [ ]:
KAPPA_VALUES = [1, 10, 100, 1000, 10000]

In [ ]:
LR_SGD = np.logspace(-4, -1, 12)
LR_MUON = np.logspace(-4, -1, 12)
LR_CLIP = np.logspace(-4, -1, 12)

## Core Functions

**Newton-Schulz:** Polar factor computation ($G \to UV^T$, all SVs = 1).

**Effective rank:** Shannon entropy-based spectral measure:
$\text{eff\_rank}(M) = \exp(-\sum_i p_i \log p_i)$ where $p_i = \sigma_i^2 / \sum \sigma_j^2$.

**Anisotropic data construction:** Creates input data $X = U \cdot \text{diag}(\sigma) \cdot Z$
where $\sigma$ are log-spaced from 1 to $1/\kappa$, giving the data matrix exact condition
number $\kappa$. Higher $\kappa$ concentrates the gradient energy in fewer directions.

**Noise fraction analysis:** Decomposes $\text{ortho}(G) = UV^T$ into signal and noise
components based on the effective rank $k$ of $G$. In the polar factor, all 32 singular
values equal 1, so the signal fraction is exactly $k/d$ and the noise fraction is $(d-k)/d$.
This is the key diagnostic: it quantifies how much of Muon's update budget goes to
non-informative directions.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, 'fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def effective_rank(M):
    sv = np.linalg.svd(M, compute_uv=False)
    sv2 = sv**2
    sv2 = sv2 / (np.sum(sv2) + 1e-30)
    sv2 = sv2[sv2 > 1e-30]
    entropy = -np.sum(sv2 * np.log(sv2))
    return np.exp(entropy)

In [ ]:
def make_anisotropic_data(kappa, seed):
    """Create data with controlled condition number."""
    rng = np.random.RandomState(seed)
    # Input data with condition number kappa
    U, _ = np.linalg.qr(rng.randn(DIM, DIM))
    sigmas = np.logspace(0, -np.log10(kappa), DIM)
    X = U @ np.diag(sigmas) @ rng.randn(DIM, BATCH_SIZE)

    W_target = rng.randn(DIM, DIM) * 0.5
    Y = W_target @ X
    return X, Y

In [ ]:
def noise_fraction_analysis(G):
    """
    Analyze how much of ortho(G)'s energy goes to noise SVs.
    Define "signal" as top-k SVs of G where k = effective_rank(G).
    """
    U, sigma, Vt = np.linalg.svd(G, full_matrices=False)
    er = int(np.round(effective_rank(G)))
    er = max(1, min(er, len(sigma)))

    # Polar factor
    Q = U @ Vt  # ortho(G) = UV^T, all SVs = 1

    # Signal: contribution of top-er right singular vectors
    # In polar factor, sigma_i = 1 for all i
    # Signal fraction = er / dim (since all SVs equal in polar factor)
    signal_frac = er / len(sigma)
    noise_frac = 1.0 - signal_frac

    # Gradient signal concentration
    grad_signal = np.sum(sigma[:er]**2) / (np.sum(sigma**2) + 1e-30)

    return signal_frac, noise_frac, er, grad_signal

## Muon-Clip: The Proposed Fix for Noise Amplification

The key innovation in this experiment: **Muon-clip(k)** zeros out the bottom $d-k$
singular values of the gradient *before* applying Newton-Schulz orthogonalization.

This means the polar factor only operates on the top-$k$ signal directions, preventing
the noise directions from being amplified to unit magnitude. If the noise amplification
theory is correct, Muon-clip should outperform standard Muon at high $\kappa$.

In [ ]:
def muon_clip_step(G, k):
    """Muon but zero out bottom (dim-k) SVs of G before orthogonalization."""
    U, sigma, Vt = np.linalg.svd(G, full_matrices=False)
    sigma_clip = sigma.copy()
    sigma_clip[k:] = 0
    G_clip = U @ np.diag(sigma_clip) @ Vt
    return newton_schulz(G_clip)

## Training Engine

Single-layer linear network: $\hat{y} = Wx$, loss $= \frac{1}{2N}\|Wx - y\|_F^2$.
Three optimizer modes: `sgd`, `muon`, and `muon_clip_k` (where $k$ is the effective rank).
Weight init: $W = I + 0.1 \cdot \mathcal{N}(0,1)$ (near-identity start).

In [ ]:
def train(X, Y, lr, opt, seed):
    rng = np.random.RandomState(seed)
    W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
    mom = np.zeros_like(W)

    for step in range(NUM_STEPS):
        pred = W @ X
        loss = 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        N = X.shape[1]
        G = (pred - Y) @ X.T / N

        if opt == 'muon':
            mom = MOMENTUM * mom + newton_schulz(G)
        elif opt == 'sgd':
            mom = MOMENTUM * mom + G
        elif opt.startswith('muon_clip_'):
            k = int(opt.split('_')[-1])
            mom = MOMENTUM * mom + muon_clip_step(G, k)

        W = W - lr * mom

    pred = W @ X
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

### Diagnostic: Gradient Spectrum and Noise Fraction at Each Kappa

Before the main experiment, let us inspect how the data condition number translates
to gradient anisotropy and noise fraction. This confirms that our experimental variable
$\kappa$ produces the intended range of gradient geometries.

In [ ]:
print("=" * 80)
print("DIAGNOSTIC: Gradient Properties at Each Data Condition Number (seed=42)")
print("=" * 80)
print(f"\n  {'kappa':>8}  {'grad_sigma1':>12}  {'grad_sigma_min':>15}  {'grad_ratio':>12}  {'eff_rank':>10}  {'noise%':>8}")
print("  " + "-" * 70)
for kappa in KAPPA_VALUES:
    X, Y = make_anisotropic_data(kappa, 42)
    rng = np.random.RandomState(42 + 5000)
    W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
    N = X.shape[1]
    G = ((W @ X - Y) @ X.T) / N
    sv = np.linalg.svd(G, compute_uv=False)
    er = effective_rank(G)
    sf, nf, _, _ = noise_fraction_analysis(G)
    print(f"  {kappa:>8}  {sv[0]:>12.6f}  {sv[-1]:>15.8f}  {sv[0]/max(sv[-1],1e-15):>12.1f}  {er:>10.1f}  {nf*100:>7.1f}%")
print(f"\n  At kappa=10000, ~{(1-effective_rank(G)/DIM)*100:.0f}% of ortho(G) energy goes to noise.")
print("  This is the fundamental problem H22b investigates.")
print("=" * 80)

---
## Main Experiment: Noise Amplification vs. Muon Advantage Across Kappa

For each data condition number $\kappa$:
1. Measure gradient effective rank and noise fraction in $\text{ortho}(G)$
2. LR sweep for SGD, Muon, and Muon-clip($k$) (12 candidates each)
3. Full training at best LR (5 seeds)
4. Compare three-way: SGD vs. Muon vs. Muon-clip

The "Clip fixes?" column in the results indicates whether Muon-clip($k$) outperforms
standard Muon by more than 20% at each $\kappa$ level.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H22b: EXTREME ANISOTROPY -- POLAR FACTOR WASTES BUDGET ON NOISE SVs")
    print("=" * 100)
    print(f"Data kappa: {KAPPA_VALUES}")
    print(f"Single layer {DIM}x{DIM}, {NUM_STEPS} steps")
    print()

    results = {}

    for kappa in KAPPA_VALUES:
        print(f"\n  kappa={kappa}")

        # Measure gradient properties
        noise_fracs = []
        eff_ranks = []
        grad_signals = []
        for s in seeds[:3]:
            X, Y = make_anisotropic_data(kappa, s)
            rng = np.random.RandomState(s + 5000)
            W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
            N = X.shape[1]
            G = ((W @ X - Y) @ X.T) / N

            sf, nf, er, gs = noise_fraction_analysis(G)
            noise_fracs.append(nf)
            eff_ranks.append(er)
            grad_signals.append(gs)

        mean_nf = np.mean(noise_fracs)
        mean_er = np.mean(eff_ranks)
        mean_gs = np.mean(grad_signals)
        print(f"    Eff rank: {mean_er:.1f}/{DIM}, Noise fraction in ortho(G): {mean_nf:.3f}, "
              f"Gradient signal concentration: {mean_gs:.3f}")

        # LR sweep for SGD, Muon, and Muon-clip
        k_clip = max(1, int(np.round(mean_er)))
        opts = [('sgd', LR_SGD), ('muon', LR_MUON), (f'muon_clip_{k_clip}', LR_CLIP)]

        best = {}
        for opt_name, grid in opts:
            best_lr, best_loss = grid[-1], float('inf')
            for lr in grid:
                losses = [train(*make_anisotropic_data(kappa, s), lr, opt_name, s+5000)
                          for s in seeds[:3]]
                ml = np.mean([l for l in losses if np.isfinite(l)] or [float('inf')])
                if ml < best_loss:
                    best_loss = ml
                    best_lr = lr
            best[opt_name] = best_lr

        # Full eval
        final = {}
        for opt_name, _ in opts:
            losses = [train(*make_anisotropic_data(kappa, s), best[opt_name], opt_name, s+5000)
                      for s in seeds]
            final[opt_name] = np.mean([l for l in losses if np.isfinite(l)] or [float('inf')])

        adv_muon = final['sgd'] / max(final['muon'], 1e-30)
        adv_clip = final['sgd'] / max(final[f'muon_clip_{k_clip}'], 1e-30)

        results[kappa] = {
            'noise_frac': mean_nf, 'eff_rank': mean_er,
            'sgd': final['sgd'], 'muon': final['muon'],
            'muon_clip': final[f'muon_clip_{k_clip}'],
            'adv_muon': adv_muon, 'adv_clip': adv_clip, 'k_clip': k_clip,
        }
        print(f"    Muon adv: {adv_muon:.1f}x, Muon-clip(k={k_clip}) adv: {adv_clip:.1f}x")

    # =========================================================================
    # RESULTS
    # =========================================================================
    print(f"\n\n{'='*100}")
    print("RESULTS: NOISE FRACTION vs MUON ADVANTAGE")
    print(f"{'='*100}")

    print(f"\n  {'kappa':>8}  {'Noise%':>8}  {'EffRank':>8}  {'Muon adv':>10}  {'Clip adv':>10}  {'Clip fixes?':>12}")
    print("  " + "-" * 60)
    for kappa in KAPPA_VALUES:
        r = results[kappa]
        fixes = "YES" if r['adv_clip'] > r['adv_muon'] * 1.2 else "NO"
        print(f"  {kappa:>8}  {r['noise_frac']*100:>7.1f}%  {r['eff_rank']:>8.1f}  "
              f"{r['adv_muon']:>10.1f}x  {r['adv_clip']:>10.1f}x  {fixes:>12}")

    # Correlation: noise fraction vs advantage
    nfs = [results[k]['noise_frac'] for k in KAPPA_VALUES]
    advs = [results[k]['adv_muon'] for k in KAPPA_VALUES]
    r_corr = np.corrcoef(nfs, np.log(np.clip(advs, 1e-10, None)))[0, 1]

    print(f"\n  Correlation(noise_fraction, log(advantage)): r = {r_corr:.3f}")

    t1 = r_corr < -0.3
    print(f"\n  T1: Higher noise fraction -> lower Muon advantage?  --> {'PASS' if t1 else 'FAIL'}  (r={r_corr:.3f})")

    # Does clipping help at high kappa?
    high_kappa = KAPPA_VALUES[-1]
    clip_helps = results[high_kappa]['adv_clip'] > results[high_kappa]['adv_muon'] * 1.2
    print(f"  T2: Muon-clip helps at kappa={high_kappa}?  --> {'PASS' if clip_helps else 'FAIL'}")

    print(f"\n{'='*100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*100}")

In [ ]:
if __name__ == '__main__':
    main()

### Interpretation of Results

**Reading the output above:**

- The results table shows noise fraction, effective rank, Muon advantage, Muon-clip
  advantage, and whether clipping helped at each $\kappa$ level.
- **T1** checks whether noise fraction negatively correlates with Muon advantage
  ($r < -0.3$). If so, the noise amplification theory is supported: as more of the update
  goes to noise directions, Muon's benefit decreases.
- **T2** checks whether Muon-clip fixes the problem at high $\kappa$. If Muon-clip
  outperforms standard Muon by > 20% at $\kappa = 10000$, the noise amplification
  mechanism is confirmed AND the proposed fix works.

**What the noise fraction tells us:**

At $\kappa = 1$, the gradient is isotropic and all 32 singular directions carry signal.
The noise fraction is near 0%, so Muon's equalization has minimal effect (advantage ~1x).

At $\kappa = 10000$, the gradient is near-rank-1. The noise fraction approaches 97%,
meaning 31 out of 32 singular directions in the orthogonalized update are pure noise.
The optimizer spends 97% of its learning rate budget moving in random directions.

**The fundamental tension:**
Muon's spectral democracy is its greatest strength AND its greatest weakness. It
democratizes all directions equally, which is great when all directions carry signal,
but catastrophic when most directions are noise.

## Conclusions

### What This Experiment Tests

**H22b** directly measures the **noise amplification failure mode** of Muon's polar factor
at extreme gradient anisotropy, and proposes a fix (Muon-clip). This is the mechanistic
companion to H22a's "sweet spot" hypothesis -- H22a shows the inverted-U shape exists,
H22b explains *why* the right tail drops off: noise equalization.

### The Three Regimes of Muon's Spectral Democracy

1. **Isotropic regime** ($\kappa \approx 1$): All SVs carry signal, equalization is
   unnecessary. Muon $\approx$ SGD. No noise amplification.
2. **Structured anisotropy** ($\kappa \sim 10{-}100$): Multiple informative directions at
   different magnitudes. Equalization helps by democratizing the update across all of them.
   Muon advantage is maximized.
3. **Extreme anisotropy** ($\kappa \geq 1000$): Gradient is near-rank-$k$. Equalization
   amplifies $d-k$ noise directions to unit magnitude. Update is dominated by noise.
   Muon advantage drops or reverses.

### Practical Implications

- If T2 passes (Muon-clip helps), this suggests a concrete improvement to Muon: apply
  a rank-$k$ truncation before Newton-Schulz. This would make Muon robust to extreme
  anisotropy while preserving its benefit at moderate anisotropy.
- The challenge is choosing $k$ adaptively -- the effective rank must be estimated from
  the gradient itself, which adds computational overhead.

### Connection to the Broader Framework

In the RG gauge-fixing interpretation, Muon removes "spectral gauge redundancy" by
equalizing singular values. H22b shows that this gauge-fixing operation has a **validity
regime**: it works when the gauge redundancy is in the *signal* subspace, but breaks
when it extends into the *noise* subspace. This is analogous to over-regularization
in physical gauge theories -- fixing a gauge that does not exist introduces artifacts.